In [ ]:
# 安装 LangChain 和 LangGraph
!pip install -U langchain langchain-openai langchain-classic

## **LangChain实现**

In [1]:
from langchain_classic import hub
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# ============== 1. 定义工具 ================
@tool
def get_weather(city: str) -> str:
  """Search for the current weather in a given city."""
  weather_db = {
        "北京": "晴，25°C，适宜出行",
        "上海": "多云，28°C，偶有阵雨",
        "深圳": "雷阵雨，30°C，注意防雷"
    }
  return weather_db.get(city, f"抱歉, 暂不支持{city}天气的查询")

@tool
def calculate(expression: str) -> str:
    """数学计算器

    Args:
        expression: 数学表达式，如 "15 + 27" 或 "100 * 0.15"
    """
    try:
        result = eval(expression)
        return f"计算结果：{expression} = {result}"
    except Exception as e:
        return f"计算错误：{str(e)}"

@tool
def search_knowledgebase(query: str) -> str:
    """查询企业知识库

    Args:
        query: 查询关键词
    """
    kb = {
        "年假政策": "员工入职满一年享有5天带薪年假，此后每增加一年加1天，上限15天",
        "报销流程": "单笔低于1000元由部门经理审批，超过1000元需财务总监签字",
        "加班调休": "工作日加班按1.5倍计算，节假日加班按3倍计算"
    }
    for key, value in kb.items():
        if key in query:
            return value
    return "未找到相关信息，请尝试其他关键词"

In [2]:
# 工具列表
tools = [get_weather, calculate, search_knowledgebase]
tools

[StructuredTool(name='get_weather', description='Search for the current weather in a given city.', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x78a97129f9c0>),
 StructuredTool(name='calculate', description='数学计算器\n\n    Args:\n        expression: 数学表达式，如 "15 + 27" 或 "100 * 0.15"', args_schema=<class 'langchain_core.utils.pydantic.calculate'>, func=<function calculate at 0x78a97100f740>),
 StructuredTool(name='search_knowledgebase', description='查询企业知识库\n\n    Args:\n        query: 查询关键词', args_schema=<class 'langchain_core.utils.pydantic.search_knowledgebase'>, func=<function search_knowledgebase at 0x78a97180e020>)]

In [14]:
!pip install -U langsmith

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.2/481.2 kB 9.3 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.8.9
    Uninstalling langsmith-0.8.9:
      Successfully uninstalled langsmith-0.8.9


In [12]:
# ================== 2. 创建 ReAct Agent =====================
from langsmith import Client

# 从 langsmith 拉取标准 ReAct 提示词模版
client = Client()
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

# 初始化LLM

from google.colab import userdata
openai_api_key = userdata.get('OPENAI_API_KEY')

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    api_key=openai_api_key,
    )

# 创建 ReAct Agent
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

# 创建执行器
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,  # 开启后可看到每一步的Thought和Action
    handle_parsing_errors=True  # 自动处理输出格式错误
)

In [15]:
# ================== 3. 运行 Agent =============================
print("=" * 50)
print("测试1：天气查询")
print("=" * 50)
result = agent_executor.invoke({
    "input": "深圳今天天气怎么样？需要带伞吗？"
})
print(f"\n最终回答：{result['output']}")

print("\n" + "=" * 50)
print("测试2：知识库查询")
print("=" * 50)
result = agent_executor.invoke({
    "input": "我想问一下咱们公司的年假是怎么计算的？"
})
print(f"\n最终回答：{result['output']}")


测试1：天气查询


> Entering new AgentExecutor chain...
为了回答这个问题，我需要查询深圳的当前天气情况，以确定是否需要带伞。  
Action: get_weather  
Action Input: 深圳  雷阵雨，30°C，注意防雷根据查询结果，深圳今天的天气是雷阵雨，温度为30°C。建议带伞以防下雨，并注意防雷。  
Final Answer: 深圳今天有雷阵雨，建议带伞并注意防雷。

> Finished chain.

最终回答：深圳今天有雷阵雨，建议带伞并注意防雷。

测试2：知识库查询


> Entering new AgentExecutor chain...
Thought: To answer the question about how the company's annual leave is calculated, I should search the company's knowledge base for relevant information.
Action: search_knowledgebase
Action Input: 年假计算未找到相关信息，请尝试其他关键词To find information about how the company's annual leave is calculated, I should try searching the knowledge base with different keywords related to annual leave or vacation policy.

Action: search_knowledgebase
Action Input: 休假政策未找到相关信息，请尝试其他关键词It seems that the specific keywords I have tried are not yielding results. I should try a broader or different approach to find the information about the company's annual leave policy.

Action: search_knowledgebase
Action 

## **LangGraph 实现**

In [17]:
# 创建 LangGraph 预构建的 ReAct Agent
from langgraph.prebuilt import create_react_agent

# 内存型状态保存器，用于保存 Agent 对话状态
from langgraph.checkpoint.memory import InMemorySaver

# 初始化聊天模型的统一入口
from langchain.chat_models import init_chat_model

# Pydantic 基础模型，用于定义结构化参数和数据校验
from pydantic import BaseModel

# 可选类型标注，例如 Optional[str] 表示 str 或 None
from typing import Optional

In [20]:
# ================== 1. 配置 LLM =========================

# Openai模型
model = init_chat_model(
    "openai:gpt-4o",
    temperature=0,
    api_key=openai_api_key
)

In [21]:
# ================== 2. 定义工具 =========================
@tool
def search_flight(origin: str, destination: str, date: str) -> str:
    """搜索航班信息

    Args:
        origin: 出发城市
        destination: 目的地城市
        date: 出发日期，格式YYYY-MM-DD
    """
    flights_db = {
        ("北京", "上海", "2026-06-01"): [
            {"航班号": "CA1234", "时间": "08:00-10:30", "价格": "¥680", "舱位": "经济舱"},
            {"航班号": "MU5678", "时间": "14:00-16:30", "价格": "¥720", "舱位": "经济舱"},
        ],
        ("上海", "深圳", "2026-06-01"): [
            {"航班号": "CZ3456", "时间": "09:30-12:00", "价格": "¥890", "舱位": "经济舱"},
            {"航班号": "HU7890", "时间": "19:00-21:30", "价格": "¥850", "舱位": "经济舱"},
        ]
    }

    key = (origin, destination, date)
    if key in flights_db:
      flights = flights_db[key]
      result = f"找到{len(flights)}个航班: \n"
      for f in flights:
        result += f"- {f['航班号']}: {f['时间']}, {f['价格']}, {f['舱位']}\n"
      return result
    return f"抱歉, 未找到{origin}到{destination}在{date}的航班"

@tool
def book_flight(flight_info: str) -> str:
    """预订航班

    Args:
        flight_info: 航班信息摘要
    """
    # 实际项目中这里调用航空公司API
    return f"✅ 预订成功！您的航班已确认。详情：{flight_info}"


@tool
def get_user_profile(user_id: str) -> str:
    """获取用户档案（验证用户身份）"""
    profiles = {
        "U001": {"姓名": "张三", "会员等级": "金卡", "常用出发地": "北京"},
        "U002": {"姓名": "李四", "会员等级": "银卡", "常用出发地": "上海"}
    }
    return str(profiles.get(user_id, {}))


In [23]:
# 工具列表
tools = [search_flight, book_flight, get_user_profile]
tools

[StructuredTool(name='search_flight', description='搜索航班信息\n\n    Args:\n        origin: 出发城市\n        destination: 目的地城市\n        date: 出发日期，格式YYYY-MM-DD', args_schema=<class 'langchain_core.utils.pydantic.search_flight'>, func=<function search_flight at 0x78a96d883ba0>),
 StructuredTool(name='book_flight', description='预订航班\n\n    Args:\n        flight_info: 航班信息摘要', args_schema=<class 'langchain_core.utils.pydantic.book_flight'>, func=<function book_flight at 0x78a96ca2f2e0>),
 StructuredTool(name='get_user_profile', description='获取用户档案（验证用户身份）', args_schema=<class 'langchain_core.utils.pydantic.get_user_profile'>, func=<function get_user_profile at 0x78a99d24cb80>)]

In [24]:
# ===================== 3. 创建带记忆的 Agent ====================

# 启用对话记忆 （关键！企业级应用必备！）
checkpointer = InMemorySaver()

agent = create_react_agent(
    model=model,
    tools=tools,
    checkpointer=checkpointer,
    prompt="""你是一个专业的机票预订助手，名字叫"飞飞"。

    服务原则：
    1. 先确认用户身份和出发地/目的地
    2. 搜索航班时提供多个选项供用户选择
    3. 用户确认后再执行预订
    4. 始终保持礼貌和专业"""
)

/tmp/ipykernel_13932/582842257.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [25]:
# ==================== 4. 执行流程 ==========================

# 模拟完整的机票预订场景
config = {"configurable": {"thread_id": "booking-001"}}

print("=" * 60)
print("🤖 飞飞：您好！我是您的机票预订助手，请问有什么可以帮您？")
print("=" * 60)

# 对话1：用户提出需求
print("\n【用户】我想6月1号从北京去深圳")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "我想6月1号从北京去深圳"}]},
    config=config
)
print(f"\n【飞飞】{result['messages'][-1].content}")

# 对话2：用户确认选择
print("\n【用户】我选CZ3456这个航班")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "我选CZ3456这个航班"}]},
    config=config
)
print(f"\n【飞飞】{result['messages'][-1].content}")

# 对话3：确认预订
print("\n【用户】确认预订")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "确认预订"}]},
    config=config
)
print(f"\n【飞飞】{result['messages'][-1].content}")

print("\n✅ 机票预订流程完成！")


🤖 飞飞：您好！我是您的机票预订助手，请问有什么可以帮您？

【用户】我想6月1号从北京去深圳

【飞飞】在为您搜索航班之前，我需要确认您的身份。请问您的用户ID是多少？

【用户】我选CZ3456这个航班

【飞飞】在为您预订航班之前，我需要确认您的身份。请问您的用户ID是多少？

【用户】确认预订

【飞飞】在为您预订航班之前，我需要确认您的身份。请问您的用户ID是多少？

✅ 机票预订流程完成！
